In [1]:
!pip install pandas openpyxl google-generativeai gradio -q

In [6]:
import pandas as pd
import gradio as gr
from google import genai
from google.genai import types

# ✅ PASTE YOUR GEMINI KEY HERE
client = genai.Client(api_key="YOUR_API_KEY")

# ---------- LAYER 1: File Loader ----------
def load_file(file_path):
    if file_path.endswith('.csv'):
        return pd.read_csv(file_path)
    elif file_path.endswith(('.xlsx', '.xls')):
        return pd.read_excel(file_path)
    elif file_path.endswith('.json'):
        return pd.read_json(file_path)
    else:
        raise ValueError("Unsupported file. Upload CSV, Excel or JSON.")

# ---------- LAYER 2: Schema Extractor ----------
def get_schema(df):
    return {
        "columns": list(df.columns),
        "dtypes": df.dtypes.astype(str).to_dict(),
        "shape": df.shape,
        "sample": df.head(3).to_dict()
    }

# ---------- LAYER 3: Prompt Builder ----------
def build_prompt(user_question, schema):
    return f"""
You are a Python data analyst. A user has uploaded a dataset.

DATASET SCHEMA:
- Columns: {schema['columns']}
- Data types: {schema['dtypes']}
- Shape: {schema['shape']} (rows, columns)
- Sample rows: {schema['sample']}

USER QUESTION: "{user_question}"

Write Python code using pandas to answer this question.

STRICT RULES:
1. The dataframe is already loaded as variable called 'df'
2. Store the final answer in a variable called 'result'
3. 'result' must be a string, number, or pandas DataFrame
4. Return ONLY executable Python code — no explanation, no markdown, no backticks
5. Handle edge cases like missing columns or empty results
"""

# ---------- LAYER 4: LLM Call ----------
def call_llm(prompt):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0,
            max_output_tokens=500
        )
    )
    code = response.text.strip()
    if code.startswith("```"):
        code = code.split("\n", 1)[1]
    if code.endswith("```"):
        code = code.rsplit("```", 1)[0]
    return code.strip()

# ---------- LAYER 5: Sandbox ----------
def execute_in_sandbox(code, df):
    safe_globals = {
        'df': df,
        'pd': pd,
        '__builtins__': {}
    }
    local_vars = {}
    try:
        exec(code, safe_globals, local_vars)
        result = local_vars.get('result', 'No result variable found')
        return {"success": True, "result": result}
    except Exception as e:
        return {"success": False, "error": str(e)}

# ---------- LAYER 6: Self Correction Loop ----------
def analyse(user_question, df):
    schema = get_schema(df)
    prompt = build_prompt(user_question, schema)
    for attempt in range(3):
        code = call_llm(prompt)
        outcome = execute_in_sandbox(code, df)
        if outcome['success']:
            return outcome['result']
        else:
            prompt += f"""
Your previous code failed with error: {outcome['error']}
Fix the code. Return ONLY corrected Python code, no markdown.
"""
    return "Could not answer after 3 attempts."

# ---------- GRADIO UI ----------
def run_analysis(file, question):
    if file is None:
        return "⚠️ Please upload a file first.", ""
    if not question.strip():
        return "⚠️ Please type a question.", ""
    try:
        df = load_file(file.name)
        result = analyse(question, df)
        if isinstance(result, pd.DataFrame):
            return result.to_markdown(), f"✅ {df.shape[0]:,} rows · {df.shape[1]} columns analysed"
        else:
            return str(result), f"✅ {df.shape[0]:,} rows · {df.shape[1]} columns analysed"
    except Exception as e:
        return f"❌ Error: {str(e)}", ""

# ---------- CUSTOM CSS ----------
css = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');

* { font-family: 'Inter', sans-serif !important; }

body, .gradio-container {
    background: #0f1117 !important;
}

.gradio-container {
    max-width: 900px !important;
    margin: 0 auto !important;
    padding: 2rem !important;
}

/* Header */
.header-box {
    background: linear-gradient(135deg, #1a1f2e 0%, #16213e 100%);
    border: 1px solid #2d3561;
    border-radius: 16px;
    padding: 2.5rem;
    text-align: center;
    margin-bottom: 1.5rem;
}

/* Cards */
.card {
    background: #1a1f2e !important;
    border: 1px solid #2d3561 !important;
    border-radius: 12px !important;
    padding: 1.5rem !important;
}

/* Inputs */
.gr-file, input, textarea {
    background: #0f1117 !important;
    border: 1px solid #2d3561 !important;
    border-radius: 8px !important;
    color: #e2e8f0 !important;
}

textarea:focus, input:focus {
    border-color: #6366f1 !important;
    box-shadow: 0 0 0 3px rgba(99,102,241,0.15) !important;
}

/* Button */
.gr-button-primary {
    background: linear-gradient(135deg, #6366f1, #8b5cf6) !important;
    border: none !important;
    border-radius: 8px !important;
    color: white !important;
    font-weight: 600 !important;
    font-size: 15px !important;
    padding: 12px 32px !important;
    cursor: pointer !important;
    transition: all 0.2s !important;
}

.gr-button-primary:hover {
    transform: translateY(-1px) !important;
    box-shadow: 0 8px 25px rgba(99,102,241,0.4) !important;
}

/* Labels */
label, .gr-block-label {
    color: #94a3b8 !important;
    font-size: 13px !important;
    font-weight: 500 !important;
    text-transform: uppercase !important;
    letter-spacing: 0.05em !important;
}

/* Output */
.output-box textarea {
    font-family: 'JetBrains Mono', monospace !important;
    font-size: 13px !important;
    line-height: 1.8 !important;
    color: #e2e8f0 !important;
}

/* Status */
.status-box textarea {
    color: #10b981 !important;
    font-size: 13px !important;
    font-weight: 500 !important;
}

/* Example pills */
.example-pill {
    display: inline-block;
    background: #1e2a3a;
    border: 1px solid #2d3561;
    border-radius: 20px;
    padding: 6px 14px;
    font-size: 12px;
    color: #94a3b8;
    margin: 4px;
}

/* Scrollbar */
::-webkit-scrollbar { width: 6px; }
::-webkit-scrollbar-track { background: #0f1117; }
::-webkit-scrollbar-thumb { background: #2d3561; border-radius: 3px; }
"""

# ---------- BUILD UI ----------
with gr.Blocks(css=css, title="QueryMind — AI Data Analyst") as app:

    # Header
    gr.HTML("""
    <div class="header-box">
        <div style="font-size:40px; margin-bottom:12px;">🧠</div>
        <h1 style="color:#e2e8f0; font-size:28px; font-weight:700; margin:0 0 8px 0; letter-spacing:-0.5px;">
            QueryMind
        </h1>
        <p style="color:#64748b; font-size:15px; margin:0 0 16px 0;">
            AI-powered data analyst · Ask questions in plain English · Get instant answers
        </p>
        <div style="display:flex; justify-content:center; gap:8px; flex-wrap:wrap;">
            <span class="example-pill">📊 CSV · Excel · JSON</span>
            <span class="example-pill">🤖 Powered by Gemini 2.5</span>
            <span class="example-pill">🔒 Sandboxed execution</span>
            <span class="example-pill">🔁 Self-correcting AI</span>
        </div>
    </div>
    """)

    # Upload + Status row
    with gr.Row():
        with gr.Column(scale=2):
            file_input = gr.File(
                label="📂 Upload Dataset",
                file_types=[".csv", ".xlsx", ".xls", ".json"],
                elem_classes=["card"]
            )
        with gr.Column(scale=1):
            status = gr.Textbox(
                label="⚡ Status",
                interactive=False,
                elem_classes=["card", "status-box"],
                lines=3
            )

    # Question input
    question_input = gr.Textbox(
        label="💬 Ask a Question",
        placeholder="e.g.  What is the total revenue per product?   |   Which region performed best?   |   Show top 5 customers by sales",
        lines=2,
        elem_classes=["card"]
    )

    # Example questions
    gr.HTML("""
    <div style="margin: 8px 0 16px 0;">
        <span style="color:#475569; font-size:12px; margin-right:8px;">Try asking:</span>
        <span class="example-pill">What is the total revenue per product?</span>
        <span class="example-pill">Which region had the highest sales?</span>
        <span class="example-pill">Show me the top 3 performing products</span>
        <span class="example-pill">What is the average order quantity?</span>
    </div>
    """)

    # Analyse button
    submit_btn = gr.Button("🔍  Analyse My Data", variant="primary", size="lg")

    # Output
    output = gr.Textbox(
        label="📊 Analysis Result",
        lines=18,
        interactive=False,
        elem_classes=["card", "output-box"]
    )

    # Footer
    gr.HTML("""
    <div style="text-align:center; margin-top:2rem; padding-top:1.5rem; border-top:1px solid #1e2a3a;">
        <p style="color:#334155; font-size:12px; margin:0;">
            Built with Python · Gemini 2.5 Flash · Pandas · Gradio &nbsp;·&nbsp;
            <span style="color:#6366f1;">QueryMind v1.0</span>
        </p>
    </div>
    """)

    submit_btn.click(
        fn=run_analysis,
        inputs=[file_input, question_input],
        outputs=[output, status]
    )

print("✅ QueryMind ready! Run Cell 3 to launch.")

/tmp/ipykernel_1414/823224152.py:224: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=css, title="QueryMind — AI Data Analyst") as app:


✅ QueryMind ready! Run Cell 3 to launch.


In [8]:
app.launch(share=True)


Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://776a92702f55a7873b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
